# EcoHome Energy Advisor: Run and Evaluate

This notebook runs realistic Berlin energy questions end to end and checks both answer quality and tool choice.

In [1]:
import json
from datetime import datetime, timedelta
from pathlib import Path

from agent import Agent


In [2]:
ECOHOME_SYSTEM_PROMPT = '''You are EcoHome's Energy Advisor for Berlin homes. Your job is to help people lower energy costs and use more of their own solar power without making their home uncomfortable.

For each question:
1. Work out what information is needed.
2. Use weather data for future solar questions and electricity prices for scheduling or cost questions.
3. Use energy history or solar history when the user asks about their past pattern.
4. Search the energy tips knowledge base when practical advice would help.
5. Use the savings calculator whenever you state a saving that can be calculated.
6. For a question with an explicit past period, query the relevant history instead of guessing.
7. Use the personalized tomorrow plan for broad next day schedules and the carbon tool when the user asks about environmental impact.
8. Give a short recommendation first, then explain the timing, cost, solar, and any assumptions in plain language.

Key capabilities: EV charging, HVAC settings, appliance scheduling, solar use, energy storage, historical analysis, personalized planning, EUR savings, and carbon estimates.

Recommendation rules:
* Give specific Berlin local hours when the data supports them.
* Prefer strong solar hours for flexible daytime tasks when the forecast supports it, but never schedule an EV after its departure time.
* Compare those hours with off peak grid prices when solar is weak.
* Mention retrieved tips with their source filename in square brackets.
* Do not claim to control a device or guarantee a saving.
* For future scheduling, use weather plus prices. For past usage, use the database. For practical steps, retrieve a tip and cite its filename.

Example questions include EV charging tomorrow, thermostat settings during a price spike, dishwasher timing, pool pump timing, and battery scheduling.'''

ecohome_agent = Agent(instructions=ECOHOME_SYSTEM_PROMPT)
print('Available tools:', ecohome_agent.get_agent_tools())


Available tools: ['get_weather_forecast', 'get_electricity_prices', 'query_energy_usage', 'query_solar_generation', 'get_recent_energy_summary', 'get_user_preferences', 'get_personalized_tomorrow_plan', 'search_energy_tips', 'calculate_energy_savings', 'calculate_carbon_impact']


In [3]:
today = datetime.now().date()
history_start = (today - timedelta(days=29)).isoformat()
history_end = today.isoformat()

test_cases = [
    {'id': 'ev_charging', 'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?', 'expected_tools': ['get_personalized_tomorrow_plan', 'get_weather_forecast', 'get_electricity_prices'], 'expected_response': {'keywords': ['solar', 'price', 'departure'], 'needs_number': True}},
    {'id': 'thermostat_peak', 'question': 'What thermostat approach should I use tomorrow afternoon if electricity prices are high?', 'expected_tools': ['get_weather_forecast', 'get_electricity_prices', 'search_energy_tips'], 'expected_response': {'keywords': ['temperature', 'peak', 'comfort'], 'needs_number': True}},
    {'id': 'dishwasher', 'question': 'How much can I save by running my dishwasher during off peak hours?', 'expected_tools': ['get_electricity_prices', 'query_energy_usage', 'calculate_energy_savings', 'search_energy_tips'], 'expected_response': {'keywords': ['dishwasher', 'eur', 'off'], 'needs_number': True}},
    {'id': 'washing_machine', 'question': 'When should I run a washing machine tomorrow in Berlin?', 'expected_tools': ['get_personalized_tomorrow_plan', 'search_energy_tips'], 'expected_response': {'keywords': ['washing', 'hour', 'price'], 'needs_number': True}},
    {'id': 'pool_pump', 'question': 'What is the best time to run my pool pump this week based on the forecast?', 'expected_tools': ['get_weather_forecast', 'get_electricity_prices', 'search_energy_tips'], 'expected_response': {'keywords': ['pool', 'solar', 'hour'], 'needs_number': True}},
    {'id': 'solar_maximization', 'question': 'How can I use more of my solar power tomorrow?', 'expected_tools': ['get_personalized_tomorrow_plan', 'get_weather_forecast', 'get_electricity_prices', 'search_energy_tips'], 'expected_response': {'keywords': ['solar', 'day', 'schedule'], 'needs_number': True}},
    {'id': 'history_reduction', 'question': 'Suggest three ways I can reduce energy use based on my usage history.', 'expected_tools': ['get_recent_energy_summary', 'search_energy_tips'], 'expected_response': {'keywords': ['ev', 'hvac', 'appliance'], 'needs_number': True}},
    {'id': 'hvac_history', 'question': f'From {history_start} through {history_end}, what does my HVAC usage history suggest about reducing evening costs?', 'expected_tools': ['query_energy_usage', 'search_energy_tips', 'calculate_energy_savings'], 'expected_response': {'keywords': ['hvac', 'evening', 'cost'], 'needs_number': True}},
    {'id': 'battery', 'question': 'How should I charge and use a home battery over the next two days?', 'expected_tools': ['get_weather_forecast', 'get_electricity_prices', 'get_user_preferences', 'search_energy_tips'], 'expected_response': {'keywords': ['battery', 'solar', 'peak'], 'needs_number': True}},
    {'id': 'solar_history', 'question': f'From {history_start} through {history_end}, how does my solar generation compare with my consumption and what should I move?', 'expected_tools': ['query_energy_usage', 'query_solar_generation', 'search_energy_tips'], 'expected_response': {'keywords': ['generation', 'consumption', 'move'], 'needs_number': True}},
    {'id': 'personalized_plan', 'question': 'Give me a tomorrow plan that respects my saved EV departure time, comfort range, and battery reserve.', 'expected_tools': ['get_personalized_tomorrow_plan'], 'expected_response': {'keywords': ['departure', 'battery', 'solar'], 'needs_number': True}},
    {'id': 'carbon_impact', 'question': 'What is the estimated CO2e impact of shifting 4 kWh of flexible use to 3 kWh of solar energy?', 'expected_tools': ['calculate_carbon_impact'], 'expected_response': {'keywords': ['co2', 'grid', 'estimate'], 'needs_number': True}},
]
assert len(test_cases) >= 10


In [4]:
def get_tool_names(messages):
    names = []
    for message in messages:
        for call in getattr(message, 'tool_calls', []) or []:
            names.append(call.get('name', 'unknown'))
        name = getattr(message, 'name', None)
        if name and message.__class__.__name__ == 'ToolMessage':
            names.append(name)
    return list(dict.fromkeys(names))

def get_tool_trace(messages):
    trace = []
    for message in messages:
        if message.__class__.__name__ != 'ToolMessage':
            continue
        content = getattr(message, 'content', '')
        if not isinstance(content, str):
            content = json.dumps(content)
        trace.append({'tool': getattr(message, 'name', 'unknown'), 'output_preview': content[:1200]})
    return trace

def final_text(messages):
    for message in reversed(messages):
        content = getattr(message, 'content', '')
        if isinstance(content, list):
            content = ''.join(
                item.get('text', '') for item in content
                if isinstance(item, dict) and item.get('type') in {'text', 'output_text'}
            )
        if message.__class__.__name__ == 'AIMessage' and isinstance(content, str) and content.strip():
            return content
    return ''

def evaluate_response(question, final_response, expected_response):
    text = final_response.lower().replace('₂', '2')
    keywords = expected_response['keywords']
    matched = [word for word in keywords if word in text]
    completeness = round(100 * len(matched) / len(keywords), 1)
    response_is_valid = bool(final_response.strip()) and 'error' not in text
    has_number = any(character.isdigit() for character in final_response)
    numeric_score = 100.0 if has_number or not expected_response['needs_number'] else 0.0
    useful_terms = ['recommend', 'schedule', 'run', 'charge', 'set', 'move', 'avoid']
    actionable_score = 100.0 if any(term in text for term in useful_terms) else 0.0
    accuracy = round((0.5 * completeness) + (0.3 * numeric_score) + (0.2 * (100.0 if response_is_valid else 0.0)), 1)
    relevance = round((0.7 * completeness) + (0.3 * actionable_score), 1)
    usefulness = round((numeric_score + actionable_score + (100.0 if response_is_valid else 0.0)) / 3, 1)
    feedback = []
    if len(matched) < len(keywords):
        feedback.append('Mention the missing ideas: ' + ', '.join(sorted(set(keywords) - set(matched))))
    if expected_response['needs_number'] and not has_number:
        feedback.append('Add a specific hour, price, or saving estimate.')
    if not response_is_valid:
        feedback.append('Return a direct answer instead of an error or empty response.')
    if not feedback:
        feedback.append('Clear, specific response with the expected ideas.')
    return {'accuracy': accuracy, 'relevance': relevance, 'completeness': completeness, 'usefulness': usefulness, 'feedback': feedback}

def evaluate_tool_usage(messages_list, expected_tools):
    actual = set(get_tool_names(messages_list))
    expected = set(expected_tools)
    matched = actual & expected
    appropriateness = round(100 * len(matched) / max(1, len(actual)), 1)
    completeness = round(100 * len(matched) / max(1, len(expected)), 1)
    feedback = []
    missing = expected - actual
    extra = actual - expected
    if missing:
        feedback.append('Missing expected tools: ' + ', '.join(sorted(missing)))
    if extra:
        feedback.append('Additional tools used: ' + ', '.join(sorted(extra)))
    if not feedback:
        feedback.append('The expected tools were used without extras.')
    return {'actual_tools': sorted(actual), 'tool_appropriateness': appropriateness, 'tool_completeness': completeness, 'feedback': feedback}

def generate_evaluation_report(results):
    response_metrics = ['accuracy', 'relevance', 'completeness', 'usefulness']
    averages = {metric: round(sum(result['response_evaluation'][metric] for result in results) / len(results), 1) for metric in response_metrics}
    averages['tool_appropriateness'] = round(sum(result['tool_evaluation']['tool_appropriateness'] for result in results) / len(results), 1)
    averages['tool_completeness'] = round(sum(result['tool_evaluation']['tool_completeness'] for result in results) / len(results), 1)
    response_score = round(sum(averages[metric] for metric in response_metrics) / len(response_metrics), 1)
    tool_score = round((averages['tool_appropriateness'] + averages['tool_completeness']) / 2, 1)
    overall_score = round((response_score * 0.65) + (tool_score * 0.35), 1)
    weak_cases = [result['test_id'] for result in results if min(result['response_evaluation']['completeness'], result['tool_evaluation']['tool_completeness']) < 80]
    strengths = [metric.replace('_', ' ') for metric, score in averages.items() if score >= 80]
    recommendations = []
    if averages['completeness'] < 80:
        recommendations.append('Add more precise timing or EUR savings where a response is incomplete.')
    if tool_score < 80:
        recommendations.append('Review missed expected tools before expanding the tool set.')
    if not recommendations:
        recommendations.append('Keep monitoring live weather and tariff assumptions as external data changes.')
    return {
        'tests_completed': len(results),
        'overall_score': overall_score,
        'response_score': response_score,
        'tool_score': tool_score,
        'average_scores': averages,
        'strengths': strengths,
        'needs_review': weak_cases,
        'recommendations': recommendations,
    }

def display_evaluation_report(report):
    print(f"Completed {report['tests_completed']} scenarios | Overall score: {report['overall_score']}/100")
    print(f"Response quality: {report['response_score']}/100 | Tool use: {report['tool_score']}/100")
    print('Strengths:', ', '.join(report['strengths']) or 'None yet')
    print('Needs review:', ', '.join(report['needs_review']) or 'None')
    print('Next steps:', ' '.join(report['recommendations']))


In [5]:
CONTEXT = 'Location: Berlin, Germany. Timezone: Europe/Berlin. Currency: EUR.'
test_results = []
for test_case in test_cases:
    print('Running:', test_case['id'])
    try:
        response = ecohome_agent.invoke(test_case['question'], CONTEXT)
        messages = response['messages']
        answer = final_text(messages)
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'final_response': answer,
            'tool_calls': get_tool_names(messages),
            'tool_trace': get_tool_trace(messages),
            'expected_tools': test_case['expected_tools'],
            'response_evaluation': evaluate_response(test_case['question'], answer, test_case['expected_response']),
            'tool_evaluation': evaluate_tool_usage(messages, test_case['expected_tools']),
            'timestamp': datetime.now().isoformat(),
        }
    except Exception as exc:
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'final_response': '',
            'tool_calls': [],
            'tool_trace': [],
            'expected_tools': test_case['expected_tools'],
            'response_evaluation': {'accuracy': 0, 'relevance': 0, 'completeness': 0, 'usefulness': 0, 'feedback': [str(exc)]},
            'tool_evaluation': {'actual_tools': [], 'tool_appropriateness': 0, 'tool_completeness': 0, 'feedback': [str(exc)]},
            'timestamp': datetime.now().isoformat(),
            'error': str(exc),
        }
    test_results.append(result)

report = generate_evaluation_report(test_results)
Path('data/test_results.json').write_text(json.dumps(test_results, indent=2), encoding='utf-8')
Path('data/evaluation_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
display_evaluation_report(report)
print(json.dumps(report, indent=2))


Running: ev_charging


Running: thermostat_peak


Running: dishwasher


Running: washing_machine


Running: pool_pump


...

/Users/autobotraos/ecohome_langgraph_rag_energy_advisor/tools.py:554: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='a7c7427a-4550-4c27-83bf-1c5af28f495e', metadata={'source': 'tip_energy_storage_optimization.txt'}, page_content='Home Battery and Energy Storage\n\nCharge a home battery when expected solar production is strong or when the grid price is low. Keep a reserve for essential loads instead of always draining the battery during the most expensive period.\n\nAvoid cycling a battery only for a tiny price difference. A useful automation needs a clear gap between charge and discharge prices after efficiency losses are considered.\n\nUse forecast based scheduling: save midday solar for the evening peak when the next day is cloudy, and leave more room for charging when a sunny day is expected.'), 0.5485217364926964), (Document(id='c054a78d-08ed-4a11-a85f-ceb1fc744305', metadata={'source': 'tip_renewable_energy_integration.txt'}, page_content='Renewable Ener

Running: solar_history


Running: personalized_plan


Running: carbon_impact


Completed 12 scenarios | Overall score: 88.1/100
Response quality: 89.3/100 | Tool use: 85.8/100
Strengths: accuracy, relevance, completeness, usefulness, tool completeness
Needs review: ev_charging, thermostat_peak, dishwasher, washing_machine, hvac_history, carbon_impact
Next steps: Keep monitoring live weather and tariff assumptions as external data changes.
{
  "tests_completed": 12,
  "overall_score": 88.1,
  "response_score": 89.3,
  "tool_score": 85.8,
  "average_scores": {
    "accuracy": 90.3,
    "relevance": 86.4,
    "completeness": 80.6,
    "usefulness": 100.0,
    "tool_appropriateness": 79.9,
    "tool_completeness": 91.7
  },
  "strengths": [
    "accuracy",
    "relevance",
    "completeness",
    "usefulness",
    "tool completeness"
  ],
  "needs_review": [
    "ev_charging",
    "thermostat_peak",
    "dishwasher",
    "washing_machine",
    "hvac_history",
    "carbon_impact"
  ],
  "recommendations": [
    "Keep monitoring live weather and tariff assumptions as e

## How to read the report

A high response score means the answer was clear, relevant, complete, and practical. A high tool score means the agent brought the right evidence into the answer. Any case listed under `needs_review` points to a response that should be improved before relying on it.